# Lesson 12A: Multi-Agent Reinforcement Learning - Theory

## Introduction

Everything so far assumed one agent acting in a stationary environment. Multi-agent RL (MARL) drops that assumption: several agents act simultaneously, and from any single agent's point of view the "environment" now includes the other agents' evolving policies.

- **Cooperative / competitive / mixed**: what do the agents' reward signals have in common?
- **Nash equilibrium**: the natural solution concept once "optimal policy" stops being well-defined for a single agent alone
- **Credit assignment**: with a shared reward, which agent's action actually caused it?
- **Independent learners vs. CTDE**: how much do agents get to see of each other during training?

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Cooperative, Competitive, and Mixed Settings

The reward structure across agents determines the whole flavor of the problem.

| Setting | Reward relationship | Example |
|---|---|---|
| **Cooperative** | Shared or aligned reward | Warehouse robots jointly minimizing delivery time |
| **Competitive (zero-sum)** | $\sum_i r_i = 0$ | Two-player board games, pursuit-evasion |
| **Mixed (general-sum)** | Independent, possibly conflicting rewards | Autonomous vehicles negotiating a junction |

The single-agent MDP generalizes to a **Markov game** (a.k.a. stochastic game): $\langle N, S, \{A_i\}, P, \{R_i\}, \gamma \rangle$ — $N$ agents, a shared state $S$, each agent's own action set $A_i$ and reward $R_i$, and transitions $P(s' \mid s, a_1, \dots, a_N)$ that depend on the **joint action**.

The core difficulty in all three settings is the same: the environment is **non-stationary** from any one agent's perspective, because the other agents are learning too. A policy that was a best response yesterday may not be one today.

## Part 2: Nash Equilibria and the Credit Assignment Problem

### Nash Equilibrium

In a single-agent MDP, "optimal" is well-defined: the policy maximizing expected return. With multiple agents optimizing simultaneously, "optimal for everyone at once" is generally impossible in competitive or mixed settings. The right solution concept is a **Nash equilibrium**: a joint policy $(\pi_1^*, \dots, \pi_N^*)$ where no agent can improve its own return by unilaterally changing its policy, holding the others fixed.

$$\forall i, \forall \pi_i: \quad J_i(\pi_i^*, \pi_{-i}^*) \geq J_i(\pi_i, \pi_{-i}^*)$$

A pure-strategy Nash equilibrium doesn't always exist (Matching Pennies has none), but a **mixed-strategy** equilibrium always does for finite games (Nash, 1950).

### The Credit Assignment Problem

In cooperative MARL with a single shared team reward $r$, every agent gets the same signal regardless of what it individually did. If the team scores, which of the five players actually deserves credit? Two common fixes:

- **Difference rewards**: give agent $i$ the counterfactual $D_i = r(s, a) - r(s, a_{-i}, c_i)$ — the actual reward minus the reward had agent $i$ taken some default action $c_i$ instead
- **Counterfactual baselines (COMA)**: a centralized critic estimates a per-agent advantage by marginalizing out that agent's action while holding the others fixed

Both isolate an individual agent's marginal contribution to a shared outcome, which a raw shared reward cannot do.

In [ ]:
# Best-response search for pure-strategy Nash equilibria in a 2-player matrix game.
# Payoff matrices: payoff[i][j] = (reward_row, reward_col) when row plays i, col plays j.

def find_pure_nash(payoff):
    n_row, n_col = len(payoff), len(payoff[0])
    equilibria = []
    for i in range(n_row):
        for j in range(n_col):
            row_payoff, col_payoff = payoff[i][j]
            row_best = all(payoff[i][j][0] >= payoff[k][j][0] for k in range(n_row))
            col_best = all(payoff[i][j][1] >= payoff[i][k][1] for k in range(n_col))
            if row_best and col_best:
                equilibria.append(((i, j), (row_payoff, col_payoff)))
    return equilibria


# Prisoner's Dilemma: actions = [Cooperate, Defect]. Mutual defection is the unique pure Nash,
# even though mutual cooperation gives both a higher payoff -- the classic cooperation dilemma.
prisoners_dilemma = [
    [(-1, -1), (-3, 0)],
    [(0, -3), (-2, -2)],
]
print("Prisoner's Dilemma pure Nash equilibria:", find_pure_nash(prisoners_dilemma))

# Matching Pennies: zero-sum, no pure-strategy Nash equilibrium exists -- only a mixed one
# (each player randomizes 50/50), which this brute-force search correctly reports as empty.
matching_pennies = [
    [(1, -1), (-1, 1)],
    [(-1, 1), (1, -1)],
]
print("Matching Pennies pure Nash equilibria:", find_pure_nash(matching_pennies))

## Part 3: Independent Q-Learning (IQL)

The simplest MARL algorithm: every agent runs single-agent Q-learning, treating the other agents as part of the environment. No communication, no shared value function — just $N$ independent learners.

**Why it can fail**: since every agent is updating its policy simultaneously, the "environment" each one sees is non-stationary — the Markov property that Q-learning's convergence proof relies on no longer holds. In practice IQL often works well enough in mildly non-stationary or cooperative settings, which is exactly why it remains a strong, cheap baseline.

In [ ]:
class IndependentQLearner:
    """Tabular Q-learning agent, unaware that it shares the environment with anyone else."""

    def __init__(self, n_actions, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.n_actions = n_actions
        self.alpha, self.gamma, self.epsilon = alpha, gamma, epsilon
        self.Q = {}

    def _q(self, state):
        return self.Q.setdefault(state, np.zeros(self.n_actions))

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self._q(state)))

    def update(self, state, action, reward, next_state):
        q = self._q(state)
        best_next = np.max(self._q(next_state))
        q[action] += self.alpha * (reward + self.gamma * best_next - q[action])


def play_repeated_game(payoff, n_rounds=2000):
    """Two independent Q-learners repeatedly play a stateless matrix game.
    State is constant, so each agent is really just learning a stationary bandit --
    except the payoffs depend on the OTHER agent's evolving policy, which is where
    the non-stationarity shows up."""
    n_actions = len(payoff)
    row_agent = IndependentQLearner(n_actions)
    col_agent = IndependentQLearner(n_actions)
    row_action_history = []

    for _ in range(n_rounds):
        state = 0  # single-state stateless game
        a_row = row_agent.act(state)
        a_col = col_agent.act(state)
        r_row, r_col = payoff[a_row][a_col]
        row_agent.update(state, a_row, r_row, state)
        col_agent.update(state, a_col, r_col, state)
        row_action_history.append(a_row)

    return row_action_history


history = play_repeated_game(prisoners_dilemma)
defect_rate = np.convolve(np.array(history) == 1, np.ones(100) / 100, mode='valid')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(defect_rate)
ax.set_xlabel('Round')
ax.set_ylabel('Row agent defection rate (window=100)')
ax.set_title('Two independent Q-learners converge toward the Nash equilibrium (mutual defection)')
plt.tight_layout()
plt.show()

## Part 4: Centralized Training, Decentralized Execution (CTDE)

CTDE is the dominant paradigm for cooperative MARL today (MADDPG, QMIX, COMA). The idea: **exploit extra information during training that won't be available at execution time**.

- **Training**: a centralized critic sees the joint state and every agent's action, so it can properly assign credit and account for non-stationarity (from the critic's point of view, nothing is non-stationary — it sees everyone's actions)
- **Execution**: each agent acts from its own local observation only, with no communication and no access to the others' actions — the policies must be **decentralizable**

This resolves the IQL non-stationarity problem during training without requiring inter-agent communication at deployment time, which is often unavailable or too slow (e.g., a swarm of drones with limited bandwidth).

In [ ]:
class CentralizedCritic:
    """Sketch: evaluates the JOINT action from the joint state -- available only at
    training time. A real implementation is a neural network; this stays tabular/toy
    to keep the training-vs-execution asymmetry visible."""

    def __init__(self, n_agents, n_actions_per_agent):
        self.n_agents = n_agents
        self.n_actions_per_agent = n_actions_per_agent
        self.Q = {}

    def value(self, joint_state, joint_action):
        key = (joint_state, tuple(joint_action))
        return self.Q.get(key, 0.0)

    def update(self, joint_state, joint_action, reward, next_value, alpha=0.1, gamma=0.9):
        key = (joint_state, tuple(joint_action))
        current = self.Q.get(key, 0.0)
        self.Q[key] = current + alpha * (reward + gamma * next_value - current)


class DecentralizedActor:
    """Sketch: acts from LOCAL observation only -- no access to other agents'
    observations or actions, matching what's available at execution time."""

    def __init__(self, n_actions):
        self.n_actions = n_actions
        self.policy = {}  # local_obs -> action probabilities

    def act(self, local_obs):
        probs = self.policy.get(local_obs, np.ones(self.n_actions) / self.n_actions)
        return np.random.choice(self.n_actions, p=probs)


# Training time: the critic sees the full joint state and joint action.
critic = CentralizedCritic(n_agents=2, n_actions_per_agent=2)
critic.update(joint_state=0, joint_action=[0, 1], reward=-1.0, next_value=0.0)
print(f"Centralized critic Q(joint_state=0, joint_action=[0,1]) = {critic.value(0, [0, 1]):.3f}")

# Execution time: each actor only ever sees its own local observation.
actor_1 = DecentralizedActor(n_actions=2)
print(f"Actor 1 acts from local_obs alone, no visibility into actor 2: action = {actor_1.act(local_obs=0)}")

## Part 5: Comparing MARL Approaches

| Approach | Training info | Execution info | Handles non-stationarity? | Handles credit assignment? |
|---|---|---|---|---|
| Independent Q-learning | Local only | Local only | No (treats others as environment) | N/A (no shared reward) |
| Joint-action learners | Full joint state/action | Full joint state/action | Yes | Yes, but doesn't scale (joint action space is exponential in $N$) |
| CTDE (e.g. MADDPG, QMIX, COMA) | Full joint state/action | Local only | Yes (via centralized critic) | Yes (centralized critic, e.g. counterfactual baselines) |

CTDE is the practical sweet spot: it gets joint-action learners' stability benefits during training while keeping deployment as cheap and decentralized as independent learners.

## Key Concepts

1. **Markov game**: the multi-agent generalization of an MDP — joint actions, per-agent rewards
2. **Non-stationarity**: every agent's environment includes the other agents' changing policies
3. **Nash equilibrium**: the right solution concept once "jointly optimal" breaks down; always exists in mixed strategies for finite games
4. **Credit assignment**: isolating an individual agent's contribution to a shared reward (difference rewards, counterfactual baselines)
5. **Independent Q-learning**: cheap, simple, ignores non-stationarity, still a strong baseline
6. **CTDE**: centralized critic at training time, decentralized actors at execution time — the dominant modern cooperative-MARL paradigm